# Hash Pipeline: Log Generation -> Cube-ID Encoding -> Decoding

이 노트북은 전체 파이프라인을 단계별로 실행합니다.

1. **Configuration** — 설정 초기화  
2. **Hash Log Generation** — 랜덤 입력 → MD5 → JSON 로그 저장  
3. **Cube-ID Image Encoding** — JSON 로그 → cube-id RGB 매핑 → PNG 저장  
4. **Image Decoding** — PNG 블록 중앙 RGB 샘플링 → CubeID byte 복원  
5. **Bulk Decode** — 전체 PNG 일괄 처리  
6. **Visualization** — 인코딩 이미지 및 cube-id 색상/바이트 결과 시각화

`cube-id`는 1 byte를 1 RGB 값으로 저장합니다.

```text
CubeID = floor(B / 64) * 64 + floor(G / 32) * 8 + floor(R / 32)
```


## 0. Environment

In [1]:
import os
import sys
import tempfile
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', os.path.join(tempfile.gettempdir(), 'matplotlib-cache'))
os.makedirs(os.environ['MPLCONFIGDIR'], exist_ok=True)

# Ensure src/ is on the path when running outside the installed package.
def find_project_root(start):
    start = Path(start).resolve()
    for path in (start, *start.parents):
        if (path / 'pyproject.toml').exists():
            return path
    return start

PROJECT_ROOT = find_project_root(os.getcwd())
_repo_src = PROJECT_ROOT / 'src'
if str(_repo_src) not in sys.path:
    sys.path.insert(0, str(_repo_src))

import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

print('Python:', sys.version.split()[0])
print('Project root:', PROJECT_ROOT)
print('Working directory:', os.getcwd())


Python: 3.12.3
Project root: /Users/choisoonwook/Experiments_local/Diffusion_HASH_inverse
Working directory: /Users/choisoonwook/Experiments_local/Diffusion_HASH_inverse/notebooks


## 1. Configuration

| 플래그 | 기본값 | 설명 |
|--------|--------|------|
| `RUN_HASH_JSON` | `True` | 입력 생성 + MD5 + JSON 로그 저장 |
| `RUN_PNG` | `True` | JSON 로그 → cube-id PNG 생성 |
| `RUN_HDF5` | `False` | PNG → HDF5 텐서 샤드 생성 |
| `ITERATION` | `10_000` | 생성할 해시 / 이미지 수 |

> 기존 로그 재사용: `RUN_HASH_JSON=False`, `RUN_PNG=True`


In [2]:
from diffusion_hash_inv.config import (
    MainConfig, HashConfig, MessageConfig, OutputConfig, Byte2RGBConfig, ImgConfig
)
from diffusion_hash_inv.main import MainEP, RuntimeConfig

# ── Run flags ─────────────────────────────────────────────────────────────
RUN_HASH_JSON = True   # JSON 로그 생성 여부
RUN_PNG       = True   # PNG 이미지 생성 여부
RUN_HDF5      = False  # HDF5 텐서 생성 여부

ITERATION = 10_000         # 생성할 해시 / 이미지 수
VERBOSE   = False

# 16 bytes = 128-bit MD5 hash output
grid_num = 4
length   = grid_num * grid_num * 8   # 128 bits

main_cfg    = MainConfig(
    verbose_flag=VERBOSE,
    clean_flag=True,
    debug_flag=False,
    make_image_flag=RUN_PNG or RUN_HDF5,
    image_workers=1,
)
message_cfg = MessageConfig(
    message_flag=False,  # False = N-bit random generation (required)
    length=length,
    random_flag=True,
    seed_flag=True,
)
hash_cfg    = HashConfig(hash_alg='MD5', length=length)
output_cfg  = OutputConfig()
rgb_cfg     = Byte2RGBConfig(encoding='cube-id', seed_flag=False, input_seed=0)

runtime_cfg = RuntimeConfig(
    main=main_cfg,
    message=message_cfg,
    hash=hash_cfg,
    output=output_cfg,
    rgb=rgb_cfg,
)
print(runtime_cfg)
print()
print('Encoding mode:', rgb_cfg.encoding)


RuntimeConfig
  MainConfig
    verbose_flag: False,
    clean_flag: True,
    debug_flag: False,
    make_image_flag: True,
    image_workers: 1,
  MessageConfig
    message_flag: False,
    length: 128,
    random_flag: True,
    seed_flag: True,
    input_seed: None,
    seed: 1445368245
  HashConfig
    hash_alg: MD5,
    length: 128,
    byteorder: little,
    word_size: 32,
    block_size: 512,
    mask: 0xFFFFFFFF,
    hierarchy: ('Step', 'Round', 'Loop')
  OutputConfig
    Root Directory: /Users/choisoonwook/Experiments_local/Diffusion_HASH_inverse,
    Data Directory: /Users/choisoonwook/Experiments_local/Diffusion_HASH_inverse/data,
    Output Directory: /Users/choisoonwook/Experiments_local/Diffusion_HASH_inverse/output,
    EMNIST Directory: /Users/choisoonwook/Experiments_local/Diffusion_HASH_inverse/EMNIST,
    Encoding: 'utf-8'
  Byte2RGBConfig
    fr_min: 0,
    fr_max: 255,
    encoding: cube-id,
    bin_width: 36 (legacy, unused by Golay encoder),
    bin_num: 7 (legac

## 2. Hash Log Generation

`MainEP.run()` 으로 입력 → MD5 → JSON 로그를 생성합니다.  
각 JSON 파일에는 **입력 메시지**, **최종 해시값**, **중간 단계 로그**가 저장됩니다.

In [3]:
main_ep = MainEP(runtime_config=runtime_cfg)

Main Entry Point Initialized.
Program Start Time: 2026-05-22 15:43:22.019577+09:00
Hash Algorithm: MD5
Message Length: 128
Data Directory: /Users/choisoonwook/Experiments_local/Diffusion_HASH_inverse/data
Output Directory: /Users/choisoonwook/Experiments_local/Diffusion_HASH_inverse/output
Clearing generated files...



In [4]:
if RUN_HASH_JSON:
    main_ep.run(
        iteration=ITERATION,
        run_hash_json=True,
        run_png=False,
        run_hdf5=False,
    )
    print('Hash/JSON generation completed.')
else:
    print('Hash/JSON generation skipped (RUN_HASH_JSON=False).')

Hash Generation Progress: 0/10000 iteration
Hash Generation Progress: 5484/10000 iteration Hash Algorithm=MD5 Message Length=128
Hash Generation Progress: 10000/10000 iteration Hash Algorithm=MD5 Message Length=128
Hash Generation Progress: done 10000/10000 iteration Hash Algorithm=MD5 Message Length=128
Hash Calculation time: 9 s, 101 ms, 796 us, 333 ns
Hash/json generation completed.

PNG image generation skipped.

HDF5 tensor generation skipped.

Total Execution Time: 9 s, 101 ms, 843 us, 375 ns

Hash/JSON generation completed.


In [5]:
# 생성된 JSON 로그 확인 (최신 run만 사용)
from diffusion_hash_inv.logger import Logs

json_dir = output_cfg.output_dir / 'json'
# 최신 타임스탬프 하위 디렉터리만 선택
_json_subdirs = sorted([d for d in json_dir.iterdir() if d.is_dir()])
_latest_json_dir = _json_subdirs[-1] if _json_subdirs else json_dir
json_files = sorted(_latest_json_dir.rglob('*.json'))
print('Latest run dir :', _latest_json_dir.name)
print('JSON log count :', len(json_files))

if json_files:
    sample = json.loads(json_files[0].read_text())
    print('\n[Sample log keys]:', list(sample.keys()))
    msg_hex = sample.get('Message', {}).get('Hex', 'N/A')
    gen_hex = sample.get('Generated hash', 'N/A')
    crt_hex = sample.get('Correct   hash', 'N/A')
    print('  message     :', msg_hex[:34], '...')
    print('  generated   :', gen_hex)
    print('  correct hash:', crt_hex)

Latest run dir : 2026-05-22 15-43-22.019577+09-00
JSON log count : 10000

[Sample log keys]: ['Metadata', 'Message', 'Generated hash', 'Correct   hash', 'Logs', 'Step Metadata']
  message     : 0x6e297bc75a14fb80200e4539016341bd ...
  generated   : 0x97c6c25422de713201528de47f643a52
  correct hash: 0x97c6c25422de713201528de47f643a52


## 3. Cube-ID Image Encoding

JSON 로그의 각 바이트를 **CubeID(0–255)** 로 보고, `Encoding Method.md`의 공식에 맞는 RGB 큐브 중심값으로 변환하여 PNG로 저장합니다.

**CubeID 공식**

```text
CubeID = floor(B / 64) * 64 + floor(G / 32) * 8 + floor(R / 32)
```

**역변환 예시**

| CubeID | R index | G index | B index | RGB center |
|--------|---------|---------|---------|------------|
| 0x00 | 0 | 0 | 0 | (16, 16, 32) |
| 0x49 | 1 | 1 | 1 | (48, 48, 96) |
| 0xff | 7 | 7 | 3 | (240, 240, 224) |

현재 PNG writer는 각 byte를 `ImgConfig.img_size`(기본 28×28 px) 블록 하나로 저장합니다. `cube-id`에서는 블록 전체가 해당 RGB 색상으로 채워지므로, 블록 중앙 픽셀만 읽어도 원 byte를 복원할 수 있습니다.


In [ ]:
if RUN_PNG:
    artifact_main_cfg = MainConfig(
        verbose_flag=main_cfg.verbose_flag,
        clean_flag=False,
        debug_flag=main_cfg.debug_flag,
        make_image_flag=True,
        image_workers=1,
    )
    artifact_runtime_cfg = RuntimeConfig(
        main=artifact_main_cfg,
        message=message_cfg,
        hash=hash_cfg,
        output=output_cfg,
        rgb=rgb_cfg,
    )
    artifact_ep = MainEP(runtime_config=artifact_runtime_cfg)
    artifact_ep.run(
        run_hash_json=False,
        run_png=True,
        run_hdf5=RUN_HDF5,
    )
    print('PNG encoding completed.')
else:
    print('PNG encoding skipped (RUN_PNG=False).')

Main Entry Point Initialized.
Program Start Time: 2026-05-22 15:43:31.198188+09:00
Hash Algorithm: MD5
Message Length: 128
Data Directory: /Users/choisoonwook/Experiments_local/Diffusion_HASH_inverse/data
Output Directory: /Users/choisoonwook/Experiments_local/Diffusion_HASH_inverse/output
Hash/json generation skipped.

RGB Image Maker Module Loaded.
RGB Image Maker Initialized.
Found 10000 logs to process.
Expected 710000 PNG files (71 images per log).
Processing Logs: 0/10000 log
Writing Images: 0/710000 image
Writing Images: 4015/710000 image
Processing Logs: 57/10000 log images=3976
Writing Images: 8052/710000 image
Processing Logs: 114/10000 log images=8023
Writing Images: 12071/710000 image
Processing Logs: 171/10000 log images=12070
Writing Images: 16185/710000 image
Processing Logs: 229/10000 log images=16188
Writing Images: 20289/710000 image
Processing Logs: 287/10000 log images=20306
Writing Images: 24358/710000 image
Processing Logs: 345/10000 log images=24424
Writing Image

In [ ]:
# 생성된 PNG 목록 확인 (최신 run만 사용)
img_root = output_cfg.data_dir / 'images'
_all_img_dirs = [d for d in img_root.iterdir() if d.is_dir()] if img_root.exists() else []
if _all_img_dirs:
    _latest_prefix = max('_'.join(d.name.split('_')[:-1]) for d in _all_img_dirs)
    png_files = sorted(
        d / 'message.png'
        for d in _all_img_dirs
        if d.name.startswith(_latest_prefix) and (d / 'message.png').exists()
    )
else:
    _latest_prefix = 'N/A'
    png_files = []

block_w, block_h = ImgConfig().img_size
print('Latest run prefix:', _latest_prefix)
print('message.png count:', len(png_files))

if png_files:
    with Image.open(png_files[0]) as im:
        w, h = im.size
    print('Sample image size (W x H):', w, 'x', h, 'px')
    print('  Block count :', w // block_w, '(=', w, 'px /', block_w, 'px/block)')


## 4. Image Decoding with Cube-ID

저장된 PNG에서 각 `ImgConfig.img_size` 블록의 중앙 RGB를 샘플링하고, 아래 공식으로 CubeID byte를 복원합니다.

```text
CubeID = floor(B / 64) * 64 + floor(G / 32) * 8 + floor(R / 32)
```

`cube-id`는 직접 양자화 매핑이므로 별도의 신뢰도나 오류정정 수를 반환하지 않습니다. 이 섹션에서는 `payload == cube_id == formula` 여부를 검증합니다.


In [ ]:
from diffusion_hash_inv.utils.image_writer import RGBImgMaker
from diffusion_hash_inv.utils.file_io import FileIO
from diffusion_hash_inv.core import RGB

# 디코더 초기화
io_ctrl   = FileIO(main_cfg, output_cfg)
img_maker = RGBImgMaker(runtime_cfg, io_ctrl, rgb_cfg)
byte2rgb  = img_maker.byte2rgb

print('Encoding mode:', byte2rgb.rgb_config.encoding)
print('Pixels/byte  :', byte2rgb.pixels_per_byte)


In [ ]:
def cube_formula(rgb: RGB) -> int:
    return (rgb.b // 64) * 64 + (rgb.g // 32) * 8 + (rgb.r // 32)


def decode_png(png_path):
    # PNG 한 장을 cube-id 방식으로 디코딩하여 byte별 record 리스트를 반환합니다.
    # 각 28x28 블록 중앙 RGB를 샘플링하고, RGB -> CubeID 공식으로 payload byte를 복원합니다.
    with Image.open(png_path) as img:
        rows = img_maker._sample_image_rgb_rows(img)

    results = []
    for row in rows:
        for rgb in row:
            cube_id = byte2rgb.rgb_to_cube_id(rgb)
            formula = cube_formula(rgb)
            results.append({
                'valid'      : cube_id is not None and cube_id == formula,
                'payload'    : cube_id,
                'cube_id'    : cube_id,
                'formula'    : formula,
                'rgb'        : rgb.as_tuple,
                'method'     : byte2rgb.rgb_config.encoding,
            })
    return results


In [ ]:
# 첫 번째 message.png 디코딩
if png_files:
    decode_results = decode_png(png_files[0])
    print('Decoded', len(decode_results), 'bytes from:', png_files[0].name)
    print()
    header = '{:>4}  {:>5}  {:>15}  {:>6}  {:>5}'.format('Byte', 'Hex', 'RGB', 'CubeID', 'Valid')
    print(header)
    print('-' * len(header))
    for idx, r in enumerate(decode_results):
        h = '0x{:02x}'.format(r['payload']) if r['payload'] is not None else 'None'
        row_str = '{:>4}  {:>5}  {:>15}  {:>6}  {:>5}'.format(
            idx, h, str(r['rgb']), str(r['cube_id']), str(r['valid'])
        )
        print(row_str)
    print()
    valid_count = sum(1 for r in decode_results if r['valid'])
    print('Valid:', valid_count, '/', len(decode_results))
else:
    print('No PNG files found. Run with RUN_PNG=True first.')


In [ ]:
# JSON 로그와 디코딩 결과 대조
if png_files and json_files:
    png_parent   = png_files[0].parent.name
    matched_json = [j for j in json_files if png_parent in str(j)]
    if not matched_json:
        matched_json = json_files[:1]

    log_data     = json.loads(matched_json[0].read_text())
    original_hex = log_data.get('Message', {}).get('Hex', '').lstrip('0x')

    decoded_bytes = bytes(
        r['payload'] for r in decode_results if r['payload'] is not None
    )
    decoded_hex = decoded_bytes.hex()

    print('Original (JSON)  :', original_hex)
    print('Decoded  (cube-id):', decoded_hex)
    print('Match            :', original_hex.lower() == decoded_hex.lower())


## 5. Bulk Decode — 전체 PNG 일괄 디코딩

발견된 모든 `message.png` 파일을 순회하여 일괄 디코딩하고 통계를 출력합니다.

In [ ]:
if png_files:
    bulk_stats = []
    for p in png_files:
        try:
            res     = decode_png(p)
            n_valid = sum(1 for r in res if r['valid'])
            formula_ok = sum(1 for r in res if r['cube_id'] == r['formula'])
            bulk_stats.append({
                'file'      : p.name,
                'bytes'     : len(res),
                'valid'     : n_valid,
                'formula_ok': formula_ok,
            })
        except Exception as exc:
            bulk_stats.append({'file': p.name, 'error': str(exc)})

    ok_stats  = [s for s in bulk_stats if 'error' not in s]
    err_stats = [s for s in bulk_stats if 'error' in s]

    print('Processed :', len(bulk_stats), '  OK:', len(ok_stats), '  Errors:', len(err_stats))

    if ok_stats:
        all_valid = all(s['valid'] == s['bytes'] for s in ok_stats)
        all_formula_ok = all(s['formula_ok'] == s['bytes'] for s in ok_stats)
        print('All bytes valid      :', all_valid)
        print('All formula checks OK:', all_formula_ok)

    if err_stats:
        for e in err_stats:
            print('  ERROR', e['file'], ':', e['error'])
else:
    print('No PNG files found.')


## 6. Visualization

### 6-1. 인코딩 이미지 원본 표시

In [ ]:
MAX_DISPLAY = 4

if png_files:
    n    = min(MAX_DISPLAY, len(png_files))
    fig, axes = plt.subplots(n, 1, figsize=(16, 2.8 * n))
    if n == 1:
        axes = [axes]

    for ax, p in zip(axes, png_files[:n]):
        with Image.open(p) as im:
            arr = np.array(im)
            title = p.parent.parent.name + '/' + p.parent.name + '/' + p.name
            title += '  (' + str(im.size[0]) + 'x' + str(im.size[1]) + ' px)'
        ax.imshow(arr)
        ax.set_title(title, fontsize=8)
        ax.axis('off')

    plt.suptitle('Cube-ID Encoded Images  (1 byte = 1 RGB block)', fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print('No PNG files to display.')


### 6-2. 바이트별 CubeID 값


In [ ]:
if png_files and 'decode_results' in dir():
    byte_idx = list(range(len(decode_results)))
    values = [r['payload'] if r['payload'] is not None else 0 for r in decode_results]
    bar_colors = ['#2ecc71' if r['valid'] else '#e74c3c' for r in decode_results]

    fig, ax = plt.subplots(1, 1, figsize=(14, 4))
    ax.bar(byte_idx, values, color=bar_colors)
    ax.set_ylim(0, 255)
    ax.set_xlabel('Byte index')
    ax.set_ylabel('CubeID / decoded byte')
    ax.set_title('Decoded CubeID Value per Byte')
    green_p = mpatches.Patch(color='#2ecc71', label='Formula match')
    red_p   = mpatches.Patch(color='#e74c3c', label='Invalid')
    ax.legend(handles=[green_p, red_p])

    plt.tight_layout()
    plt.show()

    print('Summary :', png_files[0].name)
    print('  Total bytes :', len(decode_results))
    print('  Valid bytes :', sum(r['valid'] for r in decode_results))
    print('  Min CubeID  :', min(values) if values else None)
    print('  Max CubeID  :', max(values) if values else None)
else:
    print('Run Section 4 first to populate decode_results.')


### 6-3. 블록별 Cube-ID 색상 팔레트


In [ ]:
if png_files and 'decode_results' in dir():
    with Image.open(png_files[0]) as im:
        img_np = np.array(im.convert('RGB'))

    n_bytes  = len(decode_results)
    block_w, block_h = ImgConfig().img_size

    fig, axes = plt.subplots(1, n_bytes, figsize=(max(n_bytes * 1.2, 8), 2.2))
    if n_bytes == 1:
        axes = [axes]
    fig.suptitle('Cube-ID Color Blocks  (one sampled RGB per byte)', fontsize=9)

    for i in range(n_bytes):
        y = block_h // 2
        x = i * block_w + block_w // 2
        pixel = img_np[y, x].astype(float) / 255.0

        axes[i].imshow([[pixel]])
        axes[i].axis('off')
        pay = decode_results[i]['payload']
        label = '0x{:02x}'.format(pay) if pay is not None else 'None'
        axes[i].set_title('B' + str(i) + '
' + label, fontsize=6)

    plt.tight_layout()
    plt.show()
else:
    print('Run Section 4 first to populate decode_results.')
